# NullSense — Phase 6 Training (Colab)

Before running: **Runtime > Change runtime type > T4 GPU**.

This notebook expects teammates to have uploaded their Roboflow exports to Google Drive under:
```
/MyDrive/NullSense/phase6_datasets/<category_name>/
    train/images  train/labels
    valid/images  valid/labels
    data.yaml
```
Each `<category_name>` folder is exactly what Roboflow gives you when you export a dataset as YOLOv8 — just unzip and upload as-is, don't rename the internal folders.

Run the cells in order. Dataset discovery is automatic — no need to edit this notebook when a new dataset gets uploaded.

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Pull the repo
Keeps `merge_datasets` / `train_model` / `validate_model` as a single source of truth shared with the rest of the team instead of duplicating them here.

In [ ]:
import os

REPO_DIR = '/content/NullSense'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/coderkuttan/NullSense.git {REPO_DIR}

import sys
sys.path.append(REPO_DIR)

In [ ]:
from phase6_training.train_indian_model import (
    discover_datasets, merge_datasets, train_model,
    validate_model, save_to_drive,
)

## Discover what teammates have uploaded so far

In [ ]:
DRIVE_DATASETS_DIR = '/content/drive/MyDrive/NullSense/phase6_datasets'

os.makedirs(DRIVE_DATASETS_DIR, exist_ok=True)
dataset_folders = discover_datasets(DRIVE_DATASETS_DIR)

print(f"Found {len(dataset_folders)} dataset(s):")
for f in dataset_folders:
    print(f"  - {f}")

if not dataset_folders:
    print("\nNo datasets found yet — wait for teammates to upload to")
    print(f"  {DRIVE_DATASETS_DIR}/<category_name>/")
    print("before continuing past this point.")

## Merge into one dataset

In [ ]:
assert dataset_folders, "No datasets discovered — see previous cell."

merged_dir = merge_datasets(dataset_folders, output='/content/merged_dataset')

## Train
Takes 30-60 min on a T4. `patience=10` inside `train_model` will stop early if it stops improving.

In [ ]:
best_model_path = train_model(data_yaml=f'{merged_dir}/data.yaml', epochs=50)

## Validate + save back to Drive

In [ ]:
validate_model(best_model_path, data_yaml=f'{merged_dir}/data.yaml')

In [ ]:
save_to_drive(best_model_path)

## Next steps
1. Download `nullsense_best.pt` from `/MyDrive/NullSense/` in Drive.
2. Place it in the repo at `models/nullsense_best.pt`.
3. In `shared/config.py`, set `MODEL_PATH = 'models/nullsense_best.pt'`.
4. Re-run `phase5_simulator/band_simulator.py` to confirm the new model loads and detects the Indian-specific classes.